# Cambridge Police November 2025: Check and Remove Duplicates

Reads `cambridge-police-nov-25.csv`, identifies duplicate groups using the crash timestamp, street name, latitude, and longitude, and writes `cambridge-police-nov-25-deduplicated.csv`. One row in each matching group is retained according to `KEEP_RECORD`.

## Duplicate criteria and controls

Edit `DUPLICATE_COLUMNS` to change which fields define a match, or `MIN_DUPLICATE_GROUP_SIZE` to set the smallest group considered duplicated (2 includes pairs and larger groups). Street names are trimmed and compared without regard to case; timestamps are parsed and coordinates are compared numerically.

With `REQUIRE_COMPLETE_MATCH_KEY = True`, rows missing any selected key value are not used to claim a match and are retained in the output. This avoids treating two blank coordinates or timestamps as evidence that two crashes are the same. Set `KEEP_RECORD` to `"first"` or `"last"` to choose which row to retain in each duplicate group.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_FILE = Path("cambridge-police-nov-25.csv")
OUTPUT_FILE = Path("cambridge-police-nov-25-deduplicated.csv")
DUPLICATE_COLUMNS = ["datetime", "street_name", "lat", "lng"]
MIN_DUPLICATE_GROUP_SIZE = 2
REQUIRE_COMPLETE_MATCH_KEY = True
KEEP_RECORD = "first"

## Load and inspect the input

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Could not find input file: {INPUT_FILE.resolve()}")

crashes = pd.read_csv(INPUT_FILE, dtype={"street_name": "string"})
print(f"Input records: {len(crashes):,}")
print(f"Input columns: {crashes.columns.tolist()}")
display(crashes.head())

## Build normalized match keys

The match-key frame is separate from the source data, so normalization affects comparison only; the output retains the original values.

In [ ]:
required_columns = set(DUPLICATE_COLUMNS)
missing_columns = required_columns.difference(crashes.columns)
if missing_columns:
    raise ValueError(f"Input data is missing duplicate-key columns: {sorted(missing_columns)}")
if MIN_DUPLICATE_GROUP_SIZE < 2:
    raise ValueError("MIN_DUPLICATE_GROUP_SIZE must be at least 2")
if KEEP_RECORD not in {"first", "last"}:
    raise ValueError("KEEP_RECORD must be either 'first' or 'last'")

match_keys = pd.DataFrame(index=crashes.index)
match_keys["datetime"] = pd.to_datetime(crashes["datetime"], errors="coerce")
match_keys["street_name"] = crashes["street_name"].astype("string").str.strip().str.casefold()
match_keys["lat"] = pd.to_numeric(crashes["lat"], errors="coerce")
match_keys["lng"] = pd.to_numeric(crashes["lng"], errors="coerce")

selected_keys = match_keys[DUPLICATE_COLUMNS]
if REQUIRE_COMPLETE_MATCH_KEY:
    matchable = selected_keys.notna().all(axis=1)
    if "street_name" in DUPLICATE_COLUMNS:
        matchable &= selected_keys["street_name"].ne("")
else:
    matchable = pd.Series(True, index=crashes.index)

matchable_keys = selected_keys.loc[matchable]
group_sizes = matchable_keys.groupby(DUPLICATE_COLUMNS, dropna=False).size()
duplicate_groups = (
    group_sizes[group_sizes.ge(MIN_DUPLICATE_GROUP_SIZE)]
    .rename("row_count")
    .reset_index()
    .sort_values("row_count", ascending=False)
)

duplicate_group_rows = pd.Series(False, index=crashes.index)
duplicate_rows_to_remove = pd.Series(False, index=crashes.index)
if not duplicate_groups.empty:
    group_keys = pd.MultiIndex.from_frame(duplicate_groups[DUPLICATE_COLUMNS])
    row_keys = pd.MultiIndex.from_frame(matchable_keys)
    duplicate_group_rows.loc[matchable] = row_keys.isin(group_keys)
    repeated_rows = matchable_keys.duplicated(subset=DUPLICATE_COLUMNS, keep=KEEP_RECORD)
    duplicate_rows_to_remove.loc[matchable] = repeated_rows & duplicate_group_rows.loc[matchable]

print(f"Rows eligible for matching: {int(matchable.sum()):,} of {len(crashes):,}")
print(f"Duplicate groups found: {len(duplicate_groups):,}")
print(f"Rows in duplicate groups: {int(duplicate_group_rows.sum()):,}")
print(f"Duplicate rows to remove (keeping {KEEP_RECORD}): {int(duplicate_rows_to_remove.sum()):,}")
display(duplicate_groups)

## Review matched rows

In [ ]:
duplicate_rows = crashes.loc[duplicate_group_rows].copy()
display(duplicate_rows.sort_values(DUPLICATE_COLUMNS))

## Remove repeated rows and save

In [ ]:
deduplicated = crashes.loc[~duplicate_rows_to_remove].reset_index(drop=True)
removed_count = len(crashes) - len(deduplicated)
assert removed_count == int(duplicate_rows_to_remove.sum())

deduplicated.to_csv(OUTPUT_FILE, index=False)
print(f"Removed {removed_count:,} repeated rows; retained {len(deduplicated):,} records.")
print(f"Saved deduplicated data to {OUTPUT_FILE.resolve()}")
display(deduplicated.head())